In [123]:
import torch
from torch import nn

class RMSNorm(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        eps: float = 1e-5,
        device=None,
    ):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size, device=device))
        self.eps = eps

    def forward(self, x):
        inverse_rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        normalized_x = x * inverse_rms
        return self.weight * normalized_x

def pack_hook(t):
    shape, dtype, grad_fn = t.shape, t.dtype, t.grad_fn
    print(f"Saving residual: {shape=}, {dtype=}, {grad_fn=}")
    return t

def unpack_hook(t):
    shape, dtype, grad_fn = t.shape, t.dtype, t.grad_fn
    print(f"Loading residual: {shape=}, {dtype=}, {grad_fn=}")
    return t

# device = torch.accelerator.current_accelerator()
device="cpu"
print(device)
x = torch.randn((4, 512, 2560), requires_grad=True).to(device)
rms_norm = RMSNorm(hidden_size=x.shape[-1], device=device)

with torch.autograd.graph.saved_tensors_hooks(pack_hook, unpack_hook):
    y = rms_norm(x)
    y.sum().backward()
print("====")
rms_norm2 = torch.compile(rms_norm)
with torch.autograd.graph.saved_tensors_hooks(pack_hook, unpack_hook):
    y = rms_norm2(x)
    y.sum().backward()

cpu
Saving residual: shape=torch.Size([4, 512, 2560]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([4, 512, 1]), dtype=torch.float32, grad_fn=<RsqrtBackward0 object at 0x11c5d0610>
Saving residual: shape=torch.Size([4, 512, 1]), dtype=torch.float32, grad_fn=<RsqrtBackward0 object at 0x11c5d0610>
Saving residual: shape=torch.Size([4, 512, 2560]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([4, 512, 2560]), dtype=torch.float32, grad_fn=<MulBackward0 object at 0x11c5d2260>
Saving residual: shape=torch.Size([2560]), dtype=torch.float32, grad_fn=None
Loading residual: shape=torch.Size([4, 512, 2560]), dtype=torch.float32, grad_fn=<MulBackward0 object at 0x11c5d2260>
Loading residual: shape=torch.Size([2560]), dtype=torch.float32, grad_fn=None
Loading residual: shape=torch.Size([4, 512, 1]), dtype=torch.float32, grad_fn=<RsqrtBackward0 object at 0x11c5d0610>
Loading residual: shape=torch.Size([4, 512, 2560]), dtype=torch.float32, grad_fn=None
L

In [32]:
print("d:")
print(d.grad_fn)
print(d.grad_fn.next_functions)
print(d.grad_fn.next_functions[0][0].next_functions)
print(d.grad_fn.next_functions[0][0].next_functions[0][0].next_functions)
print(
    d.grad_fn.next_functions[0][0]
    .next_functions[0][0]
    .next_functions[0][0]
    .next_functions
)

d:
((<MulBackward0 object at 0x114fd55a0>, 0), (None, 0))
((<SinBackward0 object at 0x114ea0520>, 0), (None, 0))
((<AccumulateGrad object at 0x114ea24a0>, 0),)
()


In [73]:
BATCH_SIZE = 16
DIM_IN = 1000
HIDDEN_SIZE = 100
DIM_OUT = 10


class TinyModel(torch.nn.Module):

    def __init__(self):
        super(TinyModel, self).__init__()

        self.layer1 = torch.nn.Linear(DIM_IN, HIDDEN_SIZE)
        self.relu = torch.nn.ReLU()
        self.layer2 = torch.nn.Linear(HIDDEN_SIZE, DIM_OUT)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x


some_input = torch.randn(BATCH_SIZE, DIM_IN, requires_grad=False)
ideal_output = torch.randn(BATCH_SIZE, DIM_OUT, requires_grad=False)

model = TinyModel()



In [74]:
print(model.layer2.weight[0][0:10])  # just a small slice
print(model.layer2.weight.grad)

tensor([-0.0611, -0.0337,  0.0881,  0.0398,  0.0062, -0.0523,  0.0591, -0.0799,
         0.0395, -0.0981], grad_fn=<SliceBackward0>)
None


In [75]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

prediction = model(some_input)

loss = (ideal_output - prediction).pow(2).sum()
print(loss)

tensor(182.5531, grad_fn=<SumBackward0>)


In [76]:
loss.backward()

In [91]:
x = torch.tensor(2.0, requires_grad=True)
(x * 3).backward()
x.grad = None
(x * 4).backward()
x.grad

tensor(4.)